# Minimal Workflow for Machine Learning

This notebook shows a **minimal machine learning workflow** and puts the main steps into the right context. We will use 2 different Models side by side to illustrate where steps differ depending on the model and the problem. The workflow is not meant to be a strict recipe, but rather a **general guideline** to help you structure your work and make informed decisions.

**Focus:**

- **minimal and easy-to-understand code**
- **clear structure of the workflow**
- **core steps only**

**Note:**
Machine learning workflows are **not strictly linear**.<br>Many steps can be **iterated, revisited, or performed in a different order** depending on the underlying problem.<br>Most steps in a real project are **much more detailed and complex**.<br>Here they are intentionally **simplified** to illustrate the overall workflow.

---

## Overview of Steps
**1. [Get data](#1-get-data)**
**2. [EDA](#2-eda)**
**3. [Data cleaning](#3-data-cleaning)**
**?? [Train Test Split](#3-data-cleaning)**
**4. [Preprocessing](#4-preprocessing)**
**5. [Modeling](#5-training-the-model)**
**6. [Evaluation](#6-model-evaluation)**
**7. [Iterate or Save](#7-iterateimprove-or-save)**


---

---

## 1) Get data


The data should match the **problem you want to solve**.<br>From the **target variable** we determine the **type of machine learning task**.

| Task type | Target variable |
|---|---|
| **Classification** | categorical target |
| **Regression** | continuous numerical target |

This affects later steps such as **model choice**, **evaluation metrics**, and **model interpretation**.

<details>
<summary><strong>Where to Get Data</strong></summary>

<table>
  <thead>
    <tr>
      <th>Source</th>
      <th>Examples</th>
      <th>Typical tools</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>Public datasets</strong></td>
      <td>Kaggle, UCI Machine Learning Repository, open data portals</td>
      <td>download + <code>pandas.read_csv()</code></td>
    </tr>
    <tr>
      <td><strong>Files / data exports</strong></td>
      <td>CSV, Excel, JSON, Parquet, log files</td>
      <td><code>pandas.read_csv()</code>, <code>read_excel()</code>, <code>read_json()</code></td>
    </tr>
    <tr>
      <td><strong>APIs</strong></td>
      <td>web services providing structured data</td>
      <td><code>requests</code>, <code>pandas.read_json()</code></td>
    </tr>
    <tr>
      <td><strong>Databases</strong></td>
      <td>SQL databases, data warehouses</td>
      <td><code>SQLAlchemy</code>, <code>psycopg</code>, <code>pandas.read_sql()</code></td>
    </tr>
    <tr>
      <td><strong>Company data</strong></td>
      <td>internal logs, analytics data, business databases</td>
      <td>often via database queries or internal APIs</td>
    </tr>
    <tr>
      <td><strong>Self-collected data</strong></td>
      <td>web scraping, surveys, experiments, synthetic data</td>
      <td><code>BeautifulSoup</code>, <code>Scrapy</code>, custom scripts</td>
    </tr>
  </tbody>
</table>

</details>

In [ ]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from IPython.display import display

In [ ]:
# Load the dataset
data = pd.read_csv("data/titanic.csv")

---

---

## 2) EDA

**Goal:** Understand the dataset to guide the next steps of the workflow.<br> EDA helps identify **data issues**, **useful features**, and **modeling decisions**.

| EDA helps decide | Typical tasks |
|---|---|
| which features might be useful | summary statistics |
| which cleaning steps are necessary | visualizing distributions |
| which models might be appropriate | checking correlations |
| whether data quality issues exist | identifying missing values, duplicates, inconsistencies |


<details>
<summary><strong>Example questions and tools</strong></summary>

<table>
  <thead>
    <tr>
      <th>Question</th>
      <th>Typical tools</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>What does the dataset look like?</td>
      <td><code>head()</code>, <code>info()</code>, <code>describe()</code></td>
    </tr>
    <tr>
      <td>Are there missing values?</td>
      <td><code>isna().sum()</code></td>
    </tr>
    <tr>
      <td>Are there duplicates?</td>
      <td><code>duplicated().sum()</code></td>
    </tr>
    <tr>
      <td>How is the target variable distributed?</td>
      <td><code>value_counts()</code>, <code>sns.countplot()</code></td>
    </tr>
    <tr>
      <td>How are numerical features distributed?</td>
      <td><code>hist()</code>, <code>sns.histplot()</code>, <code>sns.boxplot()</code></td>
    </tr>
    <tr>
      <td>Are features correlated?</td>
      <td><code>corr()</code>, <code>sns.heatmap()</code></td>
    </tr>
    <tr>
      <td>Are there patterns between features and target?</td>
      <td><code>groupby()</code>, <code>sns.scatterplot()</code>, <code>sns.boxplot()</code></td>
    </tr>
  </tbody>
</table>

</details>

In [ ]:
# Mini EDA. Result: PassengerId+Name not relevant for survival prediction. Some features have missing values (Age, Cabin, Embarked). 
display(data.head(3))
print("Features with missing values:\n", data.isna().sum()[data.isna().sum() > 0])

---

---

## 3) Data cleaning

**Goal:** Fix issues in the raw dataset so it can be reliably analyzed and used for modeling.  <br>Focus on correcting **data quality problems** in a **model-independent way**, not on transforming the data for a specific model yet.
<details>
<summary><strong>Typical tasks</strong></summary>

<table>
  <thead>
    <tr>
      <th>Task</th>
      <th>Typical pandas methods</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>remove irrelevant features</td>
      <td><code>drop()</code></td>
    </tr>
    <tr>
      <td>remove duplicates</td>
      <td><code>drop_duplicates()</code></td>
    </tr>
    <tr>
      <td>fix inconsistent values</td>
      <td><code>replace()</code>, <code>map()</code>, <code>str.strip()</code>, <code>str.lower()</code></td>
    </tr>
    <tr>
      <td>correct wrong data types</td>
      <td><code>astype()</code>, <code>to_datetime()</code>, <code>to_numeric()</code></td>
    </tr>
    <tr>
      <td>basic handling of missing values</td>
      <td><code>isna()</code>, <code>dropna()</code>, <code>fillna()</code></td>
    </tr>
    <tr>
      <td>basic filtering / sanity checks</td>
      <td>boolean indexing, <code>query()</code>, <code>value_counts()</code></td>
    </tr>
  </tbody>
</table>

</details>



In [ ]:
# We consider "Name" and "PassengerId" not useful for our model and drop them. 
data = data.drop(columns=["Name", "PassengerId"])

---

### Train Test Split

The moment of the split can be debated.

<details>
<summary><strong>When to split the data?</strong></summary>

<table>
  <thead>
    <tr>
      <th>Split moment</th>
      <th>Pros / Cons</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>Before EDA</strong></td>
      <td>
        <strong>Pro:</strong> strongest leakage protection<br>
        <strong>Con:</strong> little understanding of the dataset
      </td>
    </tr>
    <tr>
      <td><strong>Before data cleaning</strong></td>
      <td>
        <strong>Pro:</strong> ensures cleaning decisions are not influenced by the test set<br>
        <strong>Con:</strong> obvious data issues may remain during the split
      </td>
    </tr>
    <tr>
      <td><strong>Before preprocessing / feature engineering</strong></td>
      <td>
        <strong>Pro:</strong> most transformations can be safely fit only on training data<br>
        <strong>Con:</strong> requires more careful pipeline design
      </td>
    </tr>
    <tr>
      <td><strong>After preprocessing / feature engineering</strong></td>
      <td>
        <strong>Pro:</strong> technically easiest<br>
        <strong>Con:</strong> high risk of data leakage if transformations used the full dataset
      </td>
    </tr>
  </tbody>
</table>

</details>

<details>
<summary><strong>Why Split the data?</strong></summary>
<ul>
<li>Evaluate model performance on <strong>unseen data</strong></li>
<li>Simulate <strong>real-world predictions</strong></li>
<li>Detect <strong>overfitting</strong></li>
<li>Compare <strong>different models or hyperparameters</strong></li>
<li>Keep a <strong>final unbiased test set</strong></li>
</ul>
</details>


In [ ]:
# Separating features and target variable
y = data["Survived"]
X = data.drop(columns=["Survived"])

In [ ]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

---

---

<a id="preprocessing"></a>
## 4) Preprocessing

**Goal** Transform the cleaned data so it can be used by **your chosen model.** Focus on **model-specific transformations**. <br>**Different models** require different preprocessing, as well as **numerical** and **categorical** features. 


<details>
<summary><strong>Model choice</strong></summary>

At the latest by this step you must decide **which model to use** to build a suitable pipeline.<br>The choice depends on the **data**, the **problem type**, and the **business case** <br>(e.g. interpretability, speed, accuracy, training cost).

---

<table>
  <thead>
    <tr>
      <th>If you need / have ...</th>
      <th>Often good model choices</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>simple baseline or interpretable model</td>
      <td><strong>Linear / Logistic Regression</strong>, <strong>Decision Trees</strong></td>
    </tr>
    <tr>
      <td>small dataset or quick baseline</td>
      <td><strong>Linear / Logistic Regression</strong>, <strong>KNN</strong>, <strong>SVM</strong></td>
    </tr>
    <tr>
      <td>nonlinear patterns in structured data</td>
      <td><strong>Decision Trees</strong>, <strong>Random Forest</strong>, <strong>Extra Trees</strong></td>
    </tr>
    <tr>
      <td>strong performance on tabular data</td>
      <td><strong>Random Forest</strong>, <strong>Extra Trees</strong>, <strong>Gradient Boosting / XGBoost</strong></td>
    </tr>
    <tr>
      <td>many features or complex relationships</td>
      <td><strong>SVM</strong>, <strong>Gradient Boosting / XGBoost</strong>, <strong>MLP</strong></td>
    </tr>
    <tr>
      <td>very flexible models or larger datasets</td>
      <td><strong>MLP</strong>, <strong>Gradient Boosting / XGBoost</strong></td>
    </tr>
    <tr>
      <td>image or spatial data</td>
      <td><strong>Convolutional Neural Networks (CNNs)</strong></td>
    </tr>
  </tbody>
</table>

</details>

---

<details>
<summary><strong>Preprocessing Methods</strong></summary>

<table>
  <thead>
    <tr>
      <th>Task</th>
      <th>scikit-learn pendant</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>imputing missing values</td>
      <td><code>SimpleImputer()</code></td>
    </tr>
    <tr>
      <td>encoding categorical variables</td>
      <td><code>OneHotEncoder()</code>, <code>OrdinalEncoder()</code></td>
    </tr>
    <tr>
      <td>feature scaling</td>
      <td><code>StandardScaler()</code>, <code>MinMaxScaler()</code>, <code>RobustScaler()</code></td>
    </tr>
    <tr>
      <td>binning</td>
      <td><code>KBinsDiscretizer()</code></td>
    </tr>
    <tr>
      <td>feature transformations</td>
      <td><code>FunctionTransformer()</code>, <code>PowerTransformer()</code></td>
    </tr>
    <tr>
      <td>feature expansion</td>
      <td><code>PolynomialFeatures()</code></td>
    </tr>
    <tr>
      <td>custom preprocessing</td>
      <td><code>BaseEstimator</code>, <code>TransformerMixin</code></td>
    </tr>
    <tr>
      <td>combine steps in one workflow</td>
      <td><code>Pipeline()</code>, <code>ColumnTransformer()</code></td>
    </tr>
  </tbody>
</table>

</details>

---

<details>
<summary><strong>Model-specific preprocessing examples</strong></summary>

<table>
  <thead>
    <tr>
      <th>Model</th>
      <th>Necessary</th>
      <th>Usually unnecessary</th>
      <th>Possible benefits</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>Linear Regression / Logistic Regression</strong></td>
      <td>
        <strong>IMPUTATION</strong><br>
        <strong>ENCODING</strong><br>
        <strong>SCALING</strong> (if regularized)
      </td>
      <td>
        —
      </td>
      <td>
        binning<br>
        log transform<br>
        polynomial features<br>
        feature selection
      </td>
    </tr>
    <tr>
      <td><strong>KNN</strong></td>
      <td>
        <strong>IMPUTATION</strong><br>
        <strong>ENCODING</strong><br>
        <strong>SCALING</strong>
      </td>
      <td>
        —
      </td>
      <td>
        feature selection<br>
        dimensionality reduction
      </td>
    </tr>
    <tr>
      <td><strong>Decision Trees</strong></td>
      <td>
        <strong>IMPUTATION</strong><br>
        <strong>ENCODING</strong>
      </td>
      <td>
        scaling
      </td>
      <td>
        binning<br>
        feature selection
      </td>
    </tr>
    <tr>
      <td><strong>AdaBoost</strong></td>
      <td>
        <strong>IMPUTATION</strong><br>
        <strong>ENCODING</strong>
      </td>
      <td>
        scaling (for tree-based base estimators)
      </td>
      <td>
        feature selection<br>
        simple feature transformations
      </td>
    </tr>
  </tbody>
</table>

</details>


In [ ]:
# Identifying categorical and numerical columns -> Type: List[String]
catcols = X.select_dtypes(include=["object"]).columns
numcols = X.select_dtypes(include=["number"]).columns

# Identifying columns with missing values -> Type: List[String]
catcols_with_missing = [col for col in catcols if X[col].isna().any()]
catcols_without_missing = [col for col in catcols if col not in catcols_with_missing]

# Identifying numerical columns with and without missing values -> Type: List[String]
numcols_with_missing = [col for col in numcols if X[col].isna().any()]
numcols_without_missing = [col for col in numcols if col not in numcols_with_missing]


---

### Example A: Preprocessing pipeline Logistic Regression


<details>
<summary>Necessary Preprocessing steps for Logistic Regression</summary>

<table>
  <thead>
    <tr>
      <th>Step</th>
      <th>Reason</th>
      <th>Typical scikit-learn tool</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>IMPUTATION</strong></td>
      <td>Linear models cannot handle missing values.</td>
      <td><code>SimpleImputer()</code></td>
    </tr>
    <tr>
      <td><strong>ENCODING</strong></td>
      <td>Models require numerical input; categorical variables must be converted to numeric form.</td>
      <td><code>OneHotEncoder()</code></td>
    </tr>
    <tr>
      <td><strong>SCALING</strong></td>
      <td>Regularized linear models are sensitive to feature scale.</td>
      <td><code>StandardScaler()</code></td>
    </tr>
  </tbody>
</table>

</details>

In [ ]:
# Preprocessing for Logistic Regression 
logreg_preprocessor = ColumnTransformer(
    transformers=[
        # Preprocessing for numerical features
        ("num_missing", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())]), 
        numcols_with_missing),

        ("num", StandardScaler(), 
        numcols_without_missing),

        # Preprocessing for categorical features
        ("cat_missing", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))]),
        catcols_with_missing),

        ("cat",OneHotEncoder(handle_unknown="ignore"),
        catcols_without_missing),
        ])

---

### Example B: Preprocessing pipeline Ada Boost

<details>
<summary>Necessary Preprocessing steps for AdaBoost</summary>

<table>
  <thead>
    <tr>
      <th>Step</th>
      <th>Reason</th>
      <th>Typical scikit-learn tool</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>IMPUTATION</strong></td>
      <td>AdaBoost cannot handle missing values directly.</td>
      <td><code>SimpleImputer()</code></td>
    </tr>
    <tr>
      <td><strong>ENCODING</strong></td>
      <td>Categorical variables must be converted to numeric form.</td>
      <td><code>OneHotEncoder()</code></td>
    </tr>
    <tr>
      <td><strong>SCALING</strong></td>
      <td>Usually not necessary for tree-based base estimators.</td>
      <td>—</td>
    </tr>
  </tbody>
</table>

</details>

In [ ]:
# Preprocessing for AdaBoost
ada_preprocessor = ColumnTransformer(
    transformers=[
        # Preprocessing for numerical features
        ("num_missing", Pipeline([
                ("imputer", SimpleImputer(strategy="median"))]),
        numcols_with_missing),

        ("num", "passthrough",
        numcols_without_missing),

        # Preprocessing for categorical features
        ("cat_missing", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))]),
        catcols_with_missing),

        ("cat", OneHotEncoder(handle_unknown="ignore"),
        catcols_without_missing),
    ])

---

---

## 5) Training the model

**Goal** **Find a good configuration for your chosen model.**<br>Preprocessing made the data usable for the model. Training now focuses on **learning patterns from the training data** and you should keep the following in mind:

---

<details>
<summary><strong>Metric</strong></summary>

The **evaluation metric** should reflect the **goal of the task or business case**.<br> Since models are typically **selected and optimized using this metric**, choosing the right metric is crucial.
<table>
<thead>
<tr>
<th>Metric</th>
<th>When it is important</th>
</tr>
</thead>
<tbody>
<tr>
<td><strong>Accuracy</strong></td>
<td>balanced classification problems</td>
</tr>
<tr>
<td><strong>Precision</strong></td>
<td>when false positives are costly (e.g. spam detection)</td>
</tr>
<tr>
<td><strong>Recall</strong></td>
<td>when missing positives is costly (e.g. disease detection)</td>
</tr>
<tr>
<td><strong>F1 Score</strong></td>
<td>imbalanced classification with focus on both precision and recall</td>
</tr>
<tr>
<td><strong>ROC AUC</strong></td>
<td>ranking predictions and comparing classifiers</td>
</tr>
<tr>
<td><strong>MSE / RMSE</strong></td>
<td>regression tasks</td>
</tr>
</tbody>
</table>

</details>

---

<details>
<summary><strong>Cross-validation</strong></summary>

Cross-validation provides a **more reliable estimate of model performance**.<br>Instead of evaluating on a single split, the model is trained and evaluated on **multiple data splits**.

Typical usage in scikit-learn:

<ul>
<li><code>cross_val_score()</code></li>
<li><code>KFold</code>, <code>StratifiedKFold</code></li>
<li>already integrated in <code>GridSearchCV</code> and <code>RandomizedSearchCV</code></li>
</ul>

</details>

---
<details>
<summary><strong>Hyperparameter tuning</strong></summary>

Hyperparameters are settings chosen **before training**.<br>They control how flexible the model is, how strongly it is regularized, and how it learns from the data.<br>Tuning them helps to improve performance and control **overfitting / underfitting**.

<br>

<table>
<thead>
<tr>
<th>Model</th>
<th>Some hyperparameters</th>
<th>Main effect</th>
</tr>
</thead>
<tbody>
<tr>
<td><strong>Logistic Regression</strong></td>
<td><code>C</code>, <code>penalty</code></td>
<td><code>C</code> controls regularization strength, <code>penalty</code> sets the regularization type</td>
</tr>
<tr>
<td><strong>KNN</strong></td>
<td><code>n_neighbors</code>, <code>weights</code></td>
<td><code>n_neighbors</code> controls how local the model is, <code>weights</code> controls how neighbors influence prediction</td>
</tr>
<tr>
<td><strong>Decision Tree</strong></td>
<td><code>max_depth</code>, <code>min_samples_split</code></td>
<td>control tree complexity and help prevent overfitting</td>
</tr>
<tr>
<td><strong>AdaBoost</strong></td>
<td><code>n_estimators</code>, <code>learning_rate</code></td>
<td>control how many weak learners are added and how strongly each contributes</td>
</tr>
</tbody>
</table>

<br>

Typical tools in scikit-learn:

<ul>
<li><code>GridSearchCV</code></li>
<li><code>RandomizedSearchCV</code></li>
</ul>

</details>

---


### Example A: GridSearchCV for Logistic Regression

In [ ]:
# Combine the model and the preprocessor in one pipeline
pipe_logreg = Pipeline(
    steps=[
        ("preprocessor", logreg_preprocessor),
        ("classifier", LogisticRegression(solver="saga", max_iter=1000))
    ])

In [ ]:
# Set hyperparameters to tune
param_grid_logreg = {
    "classifier__C": [0.01, 0.1, 1],
    "classifier__l1_ratio": [0.0, 1.0],
}

# Setting the model configuration for hyperparameter tuning with GridSearchCV. 
# Cross-validation is performed internally to evaluate each hyperparameter configuration.
gs_logreg = GridSearchCV(
    pipe_logreg,
    param_grid=param_grid_logreg,
    cv=5,
    scoring="accuracy"
)

# Fitting/Training the model the data
gs_logreg.fit(X_train, y_train)

In [ ]:
# Best hyperparameters
print("Best hyperparameters:", gs_logreg.best_params_)

# Get the best model
model_best_logreg = gs_logreg.best_estimator_

---

### Example B: RandomizedSearch for AdaBoost (Less compute intensive than GridSearch)

In [ ]:
# Combine the model and the preprocessor in one pipeline
pipe_ada = Pipeline(
    steps=[
        ("preprocessor", ada_preprocessor),
        ("classifier", AdaBoostClassifier(estimator=DecisionTreeClassifier(), random_state=42))
    ])

In [ ]:
# Set hyperparameters to tune
param_grid_ada = {"classifier__n_estimators": [50, 120],
                "classifier__learning_rate": [0.1, 0.5, 1.0],
                "classifier__estimator__min_samples_split": [2, 4, 6],
                "classifier__estimator__max_depth": [1, 2, 3],
                }   

# Setting the model configuration for hyperparameter tuning with RandomizedSearchCV.
# Cross-validation is performed internally to evaluate each hyperparameter configuration.
rs_ada = RandomizedSearchCV(
    pipe_ada,
    param_distributions=param_grid_ada,
    cv=5,
    n_iter=5,
    random_state=42,
    scoring="accuracy"
)

# Fitting / training the model on the data
rs_ada.fit(X_train, y_train)

In [ ]:
# Best hyperparameters
print("Best hyperparameters:", rs_ada.best_params_)

# Get the best model
model_best_ada = rs_ada.best_estimator_

---

---

## 6) Model Evaluation

**Goal**: Evaluate how well the trained model solves the actual task. <br>Evaluation checks not only **overall performance**, but also whether the model is useful for the **business case**, generalizes well to unseen data and a lot more.
<details>
<summary><strong>What to look at</strong></summary>

<table>
  <thead>
    <tr>
      <th>Aspect</th>
      <th>What to check</th>
      <th>Typical tools</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>Metric</strong></td>
      <td>Does the metric match the task and business goal?</td>
      <td><code>accuracy_score()</code>, <code>f1_score()</code>, <code>roc_auc_score()</code>, <code>mean_squared_error()</code></td>
    </tr>
    <tr>
      <td><strong>Model comparison</strong></td>
      <td>Is this model better than baselines or alternatives?</td>
      <td><code>cross_val_score()</code>, baseline models</td>
    </tr>
    <tr>
      <td><strong>Overfitting / underfitting</strong></td>
      <td>Compare training vs. test performance</td>
      <td><code>model.score()</code>, cross-validation scores</td>
    </tr>
    <tr>
      <td><strong>Error analysis</strong></td>
      <td>Where does the model make wrong predictions?</td>
      <td><code>confusion_matrix()</code>, <code>classification_report()</code></td>
    </tr>
    <tr>
      <td><strong>Residual analysis</strong></td>
      <td>Check error patterns in regression models</td>
      <td>residual plots, predicted vs actual plots, <code>sns.residplot()</code></td>
    </tr>
    <tr>
      <td><strong>Outlier detection</strong></td>
      <td>Check unusually large errors or feature values</td>
      <td>Z-score, boxplots, IQR rule</td>
    </tr>
    <tr>
      <td><strong>Distribution checks</strong></td>
      <td>Check whether residuals follow expected distributions</td>
      <td>QQ-plots, histograms</td>
    </tr>
    <tr>
      <td><strong>Feature influence</strong></td>
      <td>Which features drive predictions?</td>
      <td><code>coef_</code>, <code>feature_importances_</code>, permutation importance</td>
    </tr>
    <tr>
      <td><strong>Generalization</strong></td>
      <td>Is performance stable on unseen data?</td>
      <td>cross-validation, test set evaluation</td>
    </tr>
  </tbody>
</table>

</details>

In [ ]:
print("Logreg Train score:", model_best_logreg.score(X_train, y_train))
print("Logreg Test score:", model_best_logreg.score(X_test, y_test))

In [ ]:
print("Ada Train score:", model_best_ada.score(X_train, y_train))
print("Ada Test score:", model_best_ada.score(X_test, y_test))

---

---

## 7) Iterate/Improve or Save 

### Option 1: Iterate & Improve the model

**Goal**: Decide what to do next based on the evaluation results.<br>After evaluating the model you typically either **improve the workflow** or **finalize the model**.

| Next step | Purpose |
|---|---|
| **More EDA** | analyze errors and understand patterns the model misses |
| **More data** | collect additional or better quality data |
| **Improve data cleaning** | fix data issues discovered during evaluation |
| **Feature engineering** | create better or more informative features |
| **Try different models** | compare alternative algorithms |
| **Hyperparameter tuning** | search a wider or more refined parameter space |
| **Improve preprocessing** | different encoding, scaling, or transformations |
| **Address class imbalance** | resampling or class weights |


### Option 2: Finalize and Save the model

<details>
<summary>Save options</summary>

You usually save the **full trained pipeline**, not only the model,<br>
so that preprocessing and prediction stay together.

<br>

| Save option | Import / command | Why use it |
|---|---|---|
| **Pickle** | `import pickle`<br>`with open("model.pkl", "wb") as f:`<br>&nbsp;&nbsp;&nbsp;&nbsp;`pickle.dump(model_best_logreg, f)` | built into Python and very simple |
| **Joblib** | `import joblib`<br>`joblib.dump(model_best_logreg, "model.joblib")` | commonly used for scikit-learn models and convenient for larger objects |

</details>

In [ ]:
import joblib

joblib.dump(model_best_logreg, "model_logreg.joblib")
joblib.dump(model_best_ada, "model_ada.joblib")

#### Load the model 

In [ ]:
model_loaded_logreg = joblib.load("model_logreg.joblib")
model_loaded_ada = joblib.load("model_ada.joblib")